# Алмазов МРТ — эксперименты: что пробовали и насколько результат устойчив
**Как запускать:** Среда выполнения → Сменить среду → **T4 GPU** → **Выполнить все**. Нужны те же файлы в Google Drive, что и для `Almazov_MRI_v3.ipynb` (архив данных и папка `almazov` с `split_v3.csv`).

Разбиение на обучение/тест **то же самое**, что в основной работе (читается из `split_v3.csv`), поэтому все цифры сравнимы с основной моделью.
Каждый эксперимент меняет **одну** вещь относительно основного рецепта. Результаты сохраняются в Drive (`almazov/experiments_v1.json`) после каждого эксперимента:
если Colab оборвался — снова «Выполнить все», готовые эксперименты пропустятся, начатый продолжится с последней эпохи.

Время на T4: примерно 3–4 часа на всё (можно в несколько заходов). Порядок — от самого важного: повторы обучения → попытки улучшить (веса классов, 30 эпох) → другие архитектуры и проверка элементов рецепта → ансамбли → сегментация.

**Честный выбор модели:** лучшая модель выбирается автоматически **по валидации** (macro-F1), тест для выбора не используется — иначе мы бы подогнали модель под тест.

In [ ]:
# === 0. НАСТРОЙКА ===
!nvidia-smi -L
!pip -q install timm segmentation-models-pytorch h5py
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, re, random, hashlib, glob, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt
cv2.setNumThreads(0)
W = Path('/content/drive/MyDrive/almazov'); W.mkdir(parents=True, exist_ok=True)
(W/'figs').mkdir(exist_ok=True); (W/'ckpt').mkdir(exist_ok=True)
ORG = Path('/content/org'); FIG = Path('/content/fig')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'Нет GPU: Среда выполнения -> Сменить среду -> T4 GPU'
CLASSES = ['Глиома', 'Менингиома', 'Гипофиз', 'Опухоли нет']   # индекс 3 = нет опухоли
print('OK, папка результатов:', W)

In [ ]:
# === 1. ДАННЫЕ ОРГАНИЗАТОРОВ: распаковка Almazov_MRI.zip из Google Drive ===
# Набор зафиксирован: 9150 из 12064 файлов (остальные источник не отдал). Пропуски распределены по всем классам.
def n_imgs(root): return sum(1 for p in Path(root).rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
if not ORG.exists() or n_imgs(ORG) < 9000:
    zs = sorted(glob.glob('/content/drive/MyDrive/**/Almazov_MRI*.zip', recursive=True))
    assert zs, 'Не нашёл Almazov_MRI.zip в Google Drive'
    z = zipfile.ZipFile(zs[0]); cnt = 0
    for info in z.infolist():
        name = info.filename
        if not (info.flag_bits & 0x800):          # zip из Windows без UTF-8: чиним кириллицу
            try: name = name.encode('cp437').decode('cp866')
            except Exception: pass
        parts = name.replace('\\', '/').split('/')
        if parts and parts[0].startswith('Almazov_MRI'): parts = parts[1:]
        if info.is_dir() or not parts or not parts[-1].lower().endswith(('.jpg', '.jpeg', '.png')): continue
        dst = ORG.joinpath(*parts); dst.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as s, open(dst, 'wb') as d: shutil.copyfileobj(s, d)
        cnt += 1
    print('Распаковано из', zs[0], ':', cnt)
print('Снимков локально:', n_imgs(ORG))

In [ ]:
# === 2. FIGSHARE (Cheng): 3064 снимка T1+контраст, маски опухолей от радиологов, ID пациентов ===
import h5py
FIGNPZ = W/'figshare256.npz'
if not FIGNPZ.exists():
    FIG.mkdir(exist_ok=True)
    for fid in [3381290, 3381293, 3381296, 3381302]:
        z = FIG/f'{fid}.zip'
        if not z.exists():
            !wget -q -O "{z}" https://ndownloader.figshare.com/files/{fid}
        zipfile.ZipFile(z).extractall(FIG/'mat')
    files = sorted(glob.glob(str(FIG/'mat'/'**'/'*.mat'), recursive=True), key=lambda s: int(Path(s).stem))
    X, M, Y, P, IDX = [], [], [], [], []
    for f in files:
        with h5py.File(f, 'r') as h:
            cj = h['cjdata']
            img = np.array(cj['image']).T.astype(np.float32)
            msk = np.array(cj['tumorMask']).T.astype(np.uint8)
            lab = int(np.array(cj['label']).flatten()[0])
            pid = ''.join(chr(int(c)) for c in np.array(cj['PID']).flatten())
        lo, hi = np.percentile(img, 0.5), np.percentile(img, 99.5)
        img = np.clip((img-lo)/(hi-lo+1e-6), 0, 1)*255
        X.append(cv2.resize(img.astype(np.uint8), (256, 256), interpolation=cv2.INTER_AREA))
        M.append(cv2.resize(msk, (256, 256), interpolation=cv2.INTER_NEAREST))
        Y.append(lab); P.append(pid); IDX.append(int(Path(f).stem))
    cheng2ours = {1: 1, 2: 0, 3: 2}     # Cheng: 1 менингиома, 2 глиома, 3 гипофиз
    np.savez_compressed(FIGNPZ, X=np.stack(X), M=np.stack(M), y=np.array([cheng2ours[v] for v in Y]),
                        pid=np.array(P), idx=np.array(IDX))
F = np.load(FIGNPZ)
fX, fM, fy, fpid = F['X'], F['M'], F['y'], F['pid'].astype(str)
print('Figshare:', fX.shape, 'пациентов:', len(set(fpid)), 'по классам:', np.bincount(fy))

In [ ]:
# === 3. ПРЕДОБРАБОТКА + АУДИТ ДАННЫХ (дубли, аугментированные копии, общие пациенты с Figshare) ===
from PIL import Image
def prep(g):
    # единая предобработка для любых источников: серый -> квадрат с чёрными полями -> 256x256 -> контраст по перцентилям
    h, w = g.shape; s = max(h, w)
    if h != w:
        c = np.zeros((s, s), g.dtype); c[(s-h)//2:(s-h)//2+h, (s-w)//2:(s-w)//2+w] = g; g = c
    g = cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA if s > 256 else cv2.INTER_LINEAR).astype(np.float32)
    lo, hi = np.percentile(g, 0.5), np.percentile(g, 99.5)
    return (np.clip((g - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

files = sorted(p for p in ORG.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
rows, imgs = [], []
for p in files:
    parts = [s.strip() for s in p.relative_to(ORG).parts]
    split = 'test' if any(s.startswith('Тест') for s in parts) else 'train'
    lab = next(s for s in parts if s in CLASSES)
    g = np.array(Image.open(p).convert('L'))
    imgs.append(prep(g))
    rows.append(dict(path=str(p), name=p.name, split=split, label=CLASSES.index(lab), h=g.shape[0], w=g.shape[1],
                     pix=hashlib.sha1(g.tobytes()).hexdigest()))
df = pd.DataFrame(rows); A = np.stack(imgs); n = len(df); LAB = df.label.values
def source(nm):
    if re.match(r'^T[re]-', nm): return 'Kaggle Tr/Te'
    if re.search(r'_(HF|VF|RO|BR|SP|DA)_\.', nm): return 'аугм. копии'
    if re.match(r'^[PGMN]_\d+\.', nm): return 'P/G/M/N_n'
    if re.match(r'^\d+\.', nm): return 'числовые'
    if re.match(r'^(gg|m|p|image)', nm, re.I): return 'SARTAJ-подобные'
    return 'Br35H/прочие'
df['source'] = df.name.map(source)
df['augbase'] = df.name.str.extract(r'^([PGMN]_\d+)(?:_[A-Z]{2}_)?\.', expand=False)
print(df.groupby(['split', 'label']).size().unstack().rename(columns=dict(enumerate(CLASSES))))
print(pd.crosstab(df.source, df.split))

# --- похожесть снимков: корреляция 64x64 с учётом 8 поворотов/отражений ---
def small(arr, s=64): return np.stack([cv2.resize(a, (s, s), interpolation=cv2.INTER_AREA) for a in arr]).astype(np.float32)
def zt(x):
    x = x.reshape(len(x), -1); x = x - x.mean(1, keepdim=True); return x / (x.norm(dim=1, keepdim=True) + 1e-6)
def d8_t(x, t):
    y = torch.rot90(x, t // 2, (1, 2)); return torch.flip(y, (2,)) if t % 2 else y
S64 = torch.tensor(small(A)); Z = zt(S64).to(dev); ZD = [zt(d8_t(S64, t)).to(dev) for t in range(8)]
I, J, C = [], [], []
for s in range(0, n, 512):
    e = min(n, s + 512)
    best = torch.stack([ZD[t][s:e] @ Z.T for t in range(8)]).max(0).values
    best[torch.arange(e - s, device=dev), torch.arange(s, e, device=dev)] = 0
    ii, jj = torch.nonzero(best >= 0.97, as_tuple=True)
    keep = (ii + s) < jj
    I.append((ii[keep] + s).cpu().numpy()); J.append(jj[keep].cpu().numpy()); C.append(best[ii[keep], jj[keep]].cpu().numpy())
I, J, C = np.concatenate(I), np.concatenate(J), np.concatenate(C)
FZt = zt(torch.tensor(small(fX))).to(dev)
fbest = np.zeros(n, np.float32); farg = np.zeros(n, int)
for s in range(0, n, 512):
    e = min(n, s + 512)
    c = torch.stack([ZD[t][s:e] @ FZt.T for t in range(8)]).max(0).values
    v, a = c.max(1); fbest[s:e] = v.cpu().numpy(); farg[s:e] = a.cpu().numpy()
fig_ok = (LAB != 3) & (fy[farg] == LAB)
del Z, ZD, FZt; torch.cuda.empty_cache()

def build(thr_dup, thr_fig, use_pid, use_names):
    par = list(range(n))
    def find(a):
        while par[a] != a:
            par[a] = par[par[a]]; a = par[a]
        return a
    def un(a, b):
        a, b = find(a), find(b)
        if a != b: par[a] = b
    for ids in df.groupby('pix').groups.values():
        ids = list(ids)
        for i in ids[1:]: un(ids[0], i)
    if use_names:
        for ids in df.dropna(subset=['augbase']).groupby(['augbase', 'label']).groups.values():
            ids = list(ids)
            for i in ids[1:]: un(ids[0], i)
    m = (C >= thr_dup) & (LAB[I] == LAB[J])
    for i, j in zip(I[m].tolist(), J[m].tolist()): un(i, j)
    pid = np.where((fbest >= thr_fig) & fig_ok, fpid[farg], '')
    if use_pid:
        first = {}
        for i in np.where(pid != '')[0].tolist():
            if pid[i] in first: un(first[pid[i]], i)
            else: first[pid[i]] = i
    return np.array([find(i) for i in range(n)]), pid

te = (df.split == 'test').values
def leak(g):
    trg = set(g[~te]); return te & np.isin(g, list(trg))
print('\nЧувствительность к порогу похожести:')
for thr in (0.97, 0.98, 0.99):
    gi, _ = build(thr, thr, False, False); gs, _ = build(thr, thr, True, False)
    print(f'  порог {thr}: тест-снимков с копией в обучении {leak(gi).sum()}/{te.sum()},'
          f' с копией или тем же пациентом {leak(gs).sum()}/{te.sum()}')
THR = 0.98
df['group'], pid = build(THR, THR, True, True)       # для разбиения: самый осторожный вариант
df['imgcomp'], _ = build(THR, THR, False, False)     # «тот же снимок» (копия, отражение, ресайз)
gstat, _ = build(THR, THR, True, False)
df['pid'] = pid; df['fig_i'] = np.where(pid != '', farg, -1)
df['leak_copy'] = leak(df.imgcomp.values); df['leak_patient'] = leak(gstat)
df['label_conflict'] = df.group.map(df.groupby('group').label.nunique() > 1).values
audit = dict(files=n, groups=int(df.group.nunique()), biggest_group=int(df.group.value_counts().max()),
             exact_dup_groups=int((df.groupby('pix').size() > 1).sum()),
             org_images_matched_to_figshare=int((pid != '').sum()), figshare_patients_found=len(set(pid) - {''}),
             test_images=int(te.sum()), test_with_copy_in_train=int(df.leak_copy.sum()),
             test_with_copy_or_same_patient_in_train=int(df.leak_patient.sum()),
             label_conflict_images=int(df.label_conflict.sum()))
print(json.dumps(audit, ensure_ascii=False, indent=1))
print(df[te].groupby('label')[['leak_copy', 'leak_patient']].mean().rename(index=dict(enumerate(CLASSES))).round(3))
# (аудит уже сохранён основным блокнотом; здесь не перезаписываем)

In [ ]:
# === 5. ЧЕСТНОЕ РАЗБИЕНИЕ: по группам «пациент / снимок», а не по картинкам ===
from sklearn.model_selection import StratifiedGroupKFold
SPLIT = W/'split_v3.csv'
if SPLIT.exists():
    sv = pd.read_csv(SPLIT, keep_default_na=False).set_index('path')
    assert set(sv.index) == set(df.path), 'Набор файлов изменился: удали split_v3.csv в Drive'
    for col in ['part', 'test_unique', 'group', 'imgcomp', 'pid', 'fig_i']:
        df[col] = df.path.map(sv[col]).values
    df['test_unique'] = df.test_unique.astype(str).eq('True'); df['pid'] = df.pid.astype(str); df['fig_i'] = df.fig_i.astype(int)
    print('Разбиение загружено из Drive (то же, что раньше).')
else:
    raise SystemExit('Нет split_v3.csv в Drive (MyDrive/almazov). Сначала нужен основной блокнот: эксперименты должны идти на том же разбиении.')
    idx = np.arange(n)
    rest, test = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(idx, LAB, df.group.values))
    tr_i, va_i = next(StratifiedGroupKFold(6, shuffle=True, random_state=SEED).split(rest, LAB[rest], df.group.values[rest]))
    part = np.array(['train'] * n, dtype=object); part[test] = 'test'; part[rest[va_i]] = 'val'
    df['part'] = part
    tm = (df.part == 'test') & (df.source != 'аугм. копии')
    df['test_unique'] = False
    df.loc[df[tm].drop_duplicates('imgcomp').index, 'test_unique'] = True
    df[['path', 'name', 'split', 'label', 'source', 'group', 'imgcomp', 'pid', 'fig_i', 'part', 'test_unique',
        'leak_copy', 'leak_patient']].to_csv(SPLIT, index=False)
    print('Разбиение создано и сохранено в Drive.')
for a_, b_ in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not (set(df.group[df.part == a_]) & set(df.group[df.part == b_])), (a_, b_)
use = ~df.label_conflict.values
TR = np.where((df.part == 'train').values & use)[0]; VA = np.where((df.part == 'val').values & use)[0]
TE = np.where((df.part == 'test').values & use)[0]; TEU = np.where(df.test_unique.values & use)[0]
print(pd.crosstab(df.part, df.label).rename(columns=dict(enumerate(CLASSES))))
print('Уникальных снимков в тесте (без копий и аугментаций):', len(TEU))
# разбиение Figshare для сегментации — по тем же пациентам
pid_part = {}
for p_, pt in zip(df.pid, df.part):
    if p_: assert pid_part.setdefault(p_, pt) == pt
fpart = np.array([pid_part.get(p_, 'train') for p_ in fpid])
s_tr, s_va, s_te = (np.where(fpart == k)[0] for k in ('train', 'val', 'test'))
print('Figshare пациентов train/val/test:', len(set(fpid[s_tr])), len(set(fpid[s_va])), len(set(fpid[s_te])),
      '| снимков:', len(s_tr), len(s_va), len(s_te))

In [ ]:
# === E1. ОБЩИЕ ФУНКЦИИ: обучение классификатора с параметрами эксперимента ===
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score
MEAN, STD = 0.449*255, 0.226*255
EXP = W/'experiments_v1.json'
R = json.load(open(EXP)) if EXP.exists() else {}
def save_R(): json.dump(R, open(EXP, 'w'), ensure_ascii=False, indent=1)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class CDS(Dataset):
    # тот же датасет, что в основном блокноте, + параметры: аугментации вкл/выкл и размер входа
    def __init__(s, idx, aug, size=256): s.idx, s.aug, s.size = np.asarray(idx), aug, size
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = s.idx[k]; x = A[i].astype(np.float32)
        if s.aug:
            if random.random() < .5: x = x[:, ::-1]
            M_ = cv2.getRotationMatrix2D((128, 128), random.uniform(-15, 15), random.uniform(.9, 1.1))
            M_[:, 2] += np.random.uniform(-12, 12, 2)
            x = cv2.warpAffine(np.ascontiguousarray(x), M_, (256, 256), borderMode=cv2.BORDER_CONSTANT)
            x = x * random.uniform(.8, 1.2) + random.uniform(-20, 20)
            if random.random() < .3: x = cv2.GaussianBlur(x, (3, 3), 0)
            if random.random() < .3: x = x + np.random.normal(0, 5, x.shape)
        if s.size != 256: x = cv2.resize(np.ascontiguousarray(x), (s.size, s.size), interpolation=cv2.INTER_AREA)
        x = (np.clip(x, 0, 255) - MEAN) / STD
        return torch.tensor(x, dtype=torch.float32)[None].repeat(3, 1, 1), int(LAB[i])

@torch.no_grad()
def cls_predict(m, idx, size=256, bs=128):
    m.eval(); P = []
    for x, _ in DataLoader(CDS(idx, False, size), bs, shuffle=False, num_workers=2, pin_memory=True):
        x = x.to(dev)
        with torch.autocast('cuda'):
            p = m(x).float().softmax(1) + m(torch.flip(x, (3,))).float().softmax(1)
        P.append((p / 2).cpu())
    return torch.cat(P).numpy()

def evaluate(P, Y):
    pr = P.argmax(1); tt = Y != 3; pt = 1 - P[:, 3]
    return dict(n=int(len(Y)), accuracy=float(accuracy_score(Y, pr)), macro_f1=float(f1_score(Y, pr, average='macro')),
                balanced_acc=float(balanced_accuracy_score(Y, pr)),
                glioma_recall=float((pr[Y == 0] == 0).mean()),
                tumor_sens=float(((pr != 3) & tt).sum() / max(tt.sum(), 1)),
                tumor_spec=float(((pr == 3) & ~tt).sum() / max((~tt).sum(), 1)),
                screen_sens_05=float(((pt >= .5) & tt).sum() / max(tt.sum(), 1)),
                screen_spec_05=float(((pt < .5) & ~tt).sum() / max((~tt).sum(), 1)),
                tumor_auc=float(roc_auc_score(tt, pt)))

PROB = W/'exp_probs'; PROB.mkdir(exist_ok=True)
def keep_probs(tag, Pva, Pte): np.savez(PROB/f'{tag}.npz', va=Pva, te=Pte)   # вероятности — для ансамбля без переобучения
def run_cls(tag, what, arch='efficientnet_b0', pretrained=True, aug=True, size=256, seed=42, epochs=15, lr=3e-4, bs=64, cw=False):
    if tag in R: print(f'[{tag}] уже готов: accuracy {R[tag]["test"]["accuracy"]:.4f}'); return
    set_seed(seed)
    m = timm.create_model(arch, pretrained=pretrained, num_classes=4).to(dev)
    dl = DataLoader(CDS(TR, aug, size), bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    opt = torch.optim.AdamW(m.parameters(), lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * len(dl), pct_start=.15)
    scaler = torch.cuda.amp.GradScaler()
    wts = None
    if cw:   # веса классов обратно пропорциональны частоте в обучении (редкие/трудные классы весят больше)
        cnt = np.bincount(LAB[TR], minlength=4).astype(np.float32); wts = torch.tensor(cnt.sum() / (4 * cnt), device=dev)
        print(f'[{tag}] веса классов:', dict(zip(CLASSES, np.round(wts.cpu().numpy(), 2))))
    crit = nn.CrossEntropyLoss(weight=wts, label_smoothing=.05)
    CK, BEST = W/'ckpt'/f'exp_{tag}_last.pt', W/'ckpt'/f'exp_{tag}_best.pt'
    start, best, best_ep = 0, -1.0, -1
    if CK.exists():
        c = torch.load(CK, map_location=dev, weights_only=False)
        m.load_state_dict(c['m']); opt.load_state_dict(c['o']); sch.load_state_dict(c['s']); scaler.load_state_dict(c['sc'])
        start, best, best_ep = c['ep'] + 1, c['best'], c.get('best_ep', -1); print(f'[{tag}] продолжаю с эпохи', start + 1)
    t_all = time.time()
    for ep in range(start, epochs):
        m.train(); t0 = time.time(); tl = 0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda'): loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()
        f1 = f1_score(LAB[VA], cls_predict(m, VA, size).argmax(1), average='macro')
        if f1 > best: best, best_ep = f1, ep; torch.save(m.state_dict(), BEST)
        torch.save(dict(m=m.state_dict(), o=opt.state_dict(), s=sch.state_dict(), sc=scaler.state_dict(), ep=ep, best=best, best_ep=best_ep), CK)
        print(f'[{tag}] эпоха {ep+1}/{epochs}: loss {tl/len(dl):.3f}  val macroF1 {f1:.4f}  лучший {best:.4f}  {time.time()-t0:.0f}с')
    m.load_state_dict(torch.load(BEST, map_location=dev))
    P = cls_predict(m, TE, size); keep_probs(tag, cls_predict(m, VA, size), P)
    R[tag] = dict(what=what, arch=arch, pretrained=pretrained, aug=aug, size=size, seed=seed, epochs=epochs, class_weights=cw,
                  params_M=round(sum(p.numel() for p in m.parameters()) / 1e6, 1), val_macro_f1=float(best), best_epoch=int(best_ep) + 1,
                  test=evaluate(P, LAB[TE]))
    save_R(); CK.unlink(missing_ok=True)          # промежуточный чекпоинт больше не нужен (экономим место в Drive)
    t = R[tag]['test']; print(f'[{tag}] ТЕСТ: accuracy {t["accuracy"]:.4f}  macroF1 {t["macro_f1"]:.4f}  чувств. {t["tumor_sens"]:.4f}  спец. {t["tumor_spec"]:.4f}')
    del m, opt; torch.cuda.empty_cache()

# основная модель (уже обучена в основном блокноте) — та же оценка тем же кодом, для сверки
if 'base' not in R:
    mb = timm.create_model('efficientnet_b0', pretrained=False, num_classes=4).to(dev)
    mb.load_state_dict(torch.load(W/'ckpt'/'cls_v3_best.pt', map_location=dev))
    Pva_b, Pte_b = cls_predict(mb, VA), cls_predict(mb, TE); keep_probs('base', Pva_b, Pte_b)
    R['base'] = dict(what='Основная модель (EfficientNet-B0, ImageNet, аугментации, 256 px)', arch='efficientnet_b0', pretrained=True,
                     aug=True, size=256, seed=42, epochs=15, val_macro_f1=float(f1_score(LAB[VA], Pva_b.argmax(1), average='macro')),
                     test=evaluate(Pte_b, LAB[TE]))
    save_R(); del mb; torch.cuda.empty_cache()
if (W/'results_cls_v3.json').exists():   # сверка: должно совпасть с основным блокнотом
    ref = json.load(open(W/'results_cls_v3.json'))['test_group']['accuracy']
    print(f'Сверка с основным блокнотом: accuracy {R["base"]["test"]["accuracy"]:.4f} (было {ref:.4f})',
          'OK' if abs(R['base']['test']['accuracy'] - ref) < 0.003 else '<- НЕ СОВПАДАЕТ, напиши мне')
def safe(fn, *a, **k):
    # если один эксперимент упал (например, не хватило памяти) — остальные всё равно выполнятся
    try: fn(*a, **k)
    except Exception as e:
        print(f'!!! эксперимент {a[0]} упал: {type(e).__name__}: {e}'); torch.cuda.empty_cache()
print('Основная модель на честном тесте:', {k: round(v, 4) for k, v in R['base']['test'].items() if k != 'n'})

In [ ]:
# === E2. ЭКСПЕРИМЕНТЫ С КЛАССИФИКАТОРОМ (каждый меняет одну вещь) ===
# 1) Повторы обучения с другим seed: насколько результат зависит от случайности (+ из них соберём ансамбль)
safe(run_cls, 'seed1', 'Тот же рецепт, повтор обучения (seed 1)', seed=1)
safe(run_cls, 'seed2', 'Тот же рецепт, повтор обучения (seed 2)', seed=2)
# 2) Попытки улучшить (в первую очередь — глиомы)
safe(run_cls, 'class_w', 'Веса классов в функции потерь (против путаницы глиома/менингиома)', cw=True)
safe(run_cls, 'epochs30', 'Дольше обучение: 30 эпох вместо 15', epochs=30)
# 3) Что даёт каждый элемент рецепта
safe(run_cls, 'no_aug', 'Без аугментаций (поворотов, отражений, яркости, шума)', aug=False)
safe(run_cls, 'resnet50', 'Другая архитектура: ResNet-50 (в 6 раз больше параметров)', arch='resnet50')
safe(run_cls, 'effb3', 'Другая архитектура: EfficientNet-B3 (батч 32 — иначе не влезает в память)', arch='efficientnet_b3', bs=32)
safe(run_cls, 'no_pretrain', 'Без предобучения на ImageNet (с нуля)', pretrained=False)
safe(run_cls, 'size128', 'Меньшее разрешение входа: 128×128 вместо 256×256', size=128)

In [ ]:
# === E2b. АНСАМБЛЬ: среднее вероятностей нескольких моделей (без нового обучения) ===
def ensemble(tag, members, what):
    have = [k for k in members if (PROB/f'{k}.npz').exists()]
    if len(have) < 2: print(f'[{tag}] пропускаю: готово моделей {len(have)}'); return
    Z = [np.load(PROB/f'{k}.npz') for k in have]
    Pva_, Pte_ = np.mean([z['va'] for z in Z], 0), np.mean([z['te'] for z in Z], 0)
    R[tag] = dict(what=what + f' ({len(have)} шт.: ' + ', '.join(have) + ')', members=have,
                  val_macro_f1=float(f1_score(LAB[VA], Pva_.argmax(1), average='macro')), test=evaluate(Pte_, LAB[TE]))
    save_R(); t = R[tag]['test']
    print(f'[{tag}] val macroF1 {R[tag]["val_macro_f1"]:.4f} | ТЕСТ: accuracy {t["accuracy"]:.4f}  macroF1 {t["macro_f1"]:.4f}  recall глиомы {t["glioma_recall"]:.4f}')
ensemble('ens_seeds', ['base', 'seed1', 'seed2'], 'Ансамбль: основная модель + 2 повтора')
ensemble('ens_all', ['base', 'seed1', 'seed2', 'class_w', 'epochs30', 'resnet50', 'effb3'], 'Ансамбль всех сильных вариантов')

# === ВЫБОР ЛУЧШЕЙ МОДЕЛИ — ТОЛЬКО ПО ВАЛИДАЦИИ (тест не используется для выбора) ===
cand = {k: v for k, v in R.items() if isinstance(v, dict) and 'val_macro_f1' in v and 'test' in v and not k.startswith('seg') and k != 'selected'}
best_k = max(cand, key=lambda k: cand[k]['val_macro_f1'])
R['selected'] = dict(tag=best_k, rule='максимальный macro-F1 на валидации', val_macro_f1=cand[best_k]['val_macro_f1'],
                     test=cand[best_k]['test'], base_test=R['base']['test']); save_R()
print(pd.DataFrame({k: dict(val_F1=v['val_macro_f1'], test_acc=v['test']['accuracy'], test_F1=v['test']['macro_f1'],
                            glioma=v['test']['glioma_recall']) for k, v in cand.items()}).T.sort_values('val_F1', ascending=False).round(4))
print(f'\nВЫБРАНА ПО ВАЛИДАЦИИ: {best_k}. Тест: accuracy {cand[best_k]["test"]["accuracy"]:.4f} '
      f'(основная было {R["base"]["test"]["accuracy"]:.4f}), recall глиомы {cand[best_k]["test"]["glioma_recall"]:.4f} '
      f'(было {R["base"]["test"]["glioma_recall"]:.4f})')

In [ ]:
# === E3. ЭКСПЕРИМЕНТЫ С СЕГМЕНТАЦИЕЙ (U-Net ResNet34, тот же рецепт, что в основном блокноте) ===
import segmentation_models_pytorch as smp
def d8(x, t):
    y = np.rot90(x, t // 2); return np.fliplr(y) if t % 2 else y
def z1(v):
    v = v.ravel() - v.mean(); return v / (np.linalg.norm(v) + 1e-6)
fs64 = small(fX); a64 = small(A)
om_idx = np.where(df.pid.values != '')[0]
OM = np.zeros((len(om_idx), 256, 256), np.uint8)
for q, i in enumerate(om_idx):
    fi = int(df.fig_i.values[i]); a_ = z1(a64[i])
    t = int(np.argmax([float(a_ @ z1(d8(fs64[fi], t).copy())) for t in range(8)])); OM[q] = d8(fM[fi], t).copy()
part = df.part.values
q_tr = [q for q, i in enumerate(om_idx) if part[i] == 'train']; q_te = [q for q, i in enumerate(om_idx) if part[i] == 'test']
ZERO = np.zeros((256, 256), np.uint8)
def get_xm(src, i):
    if src == 'f': return fX[i], fM[i]
    if src == 'o': return A[om_idx[i]], OM[i]
    return A[i], ZERO
class SDS(Dataset):
    def __init__(s, items, aug): s.items, s.aug = items, aug
    def __len__(s): return len(s.items)
    def __getitem__(s, k):
        x, m = get_xm(*s.items[k]); x = x.astype(np.float32); m = m.astype(np.float32)
        if s.aug:
            if random.random() < .5: x, m = x[:, ::-1], m[:, ::-1]
            if random.random() < .5: x, m = x[::-1], m[::-1]
            M_ = cv2.getRotationMatrix2D((128, 128), random.uniform(-20, 20), random.uniform(.85, 1.15))
            x = cv2.warpAffine(np.ascontiguousarray(x), M_, (256, 256))
            m = cv2.warpAffine(np.ascontiguousarray(m), M_, (256, 256), flags=cv2.INTER_NEAREST)
            x = x * random.uniform(.8, 1.2) + random.uniform(-20, 20)
        x = (np.clip(x, 0, 255) - MEAN) / STD
        return torch.tensor(x, dtype=torch.float32)[None], torch.tensor(np.ascontiguousarray(m), dtype=torch.float32)[None]
def arrays(items):
    xm = [get_xm(s_, i) for s_, i in items]
    return np.stack([a for a, _ in xm]), np.stack([b for _, b in xm])
def dice_np(p, m):
    inter = (p & m).sum((1, 2)); den = p.sum((1, 2)) + m.sum((1, 2))
    return (2 * inter + 1e-6) / (den + 1e-6)
@torch.no_grad()
def seg_probs(net, xs):
    net.eval(); out = []
    for s in range(0, len(xs), 32):
        x = torch.tensor((xs[s:s+32].astype(np.float32) - MEAN) / STD)[:, None].to(dev)
        with torch.autocast('cuda'):
            p = torch.sigmoid(net(x).float()) + torch.flip(torch.sigmoid(net(torch.flip(x, (3,))).float()), (3,))
        out.append((p / 2).cpu().numpy()[:, 0])
    return np.concatenate(out) if out else np.zeros((0, 256, 256), np.float32)
def demo_contour(pm):
    # правило прототипа: порог 0.5 и выброс пятен меньше 30 пикселей
    m = (pm > .5).astype(np.uint8); k, lab, st, _ = cv2.connectedComponentsWithStats(m)
    for c in range(1, k):
        if st[c, cv2.CC_STAT_AREA] < 30: m[lab == c] = 0
    return m.sum() > 0

XV, MV = arrays([('f', i) for i in s_va]); XF, MF = arrays([('f', i) for i in s_te])
XO, MO = arrays([('o', q) for q in q_te]) if q_te else (None, None)
NT = [i for i in TE if LAB[i] == 3]; XN = A[NT]
def seg_eval(net):
    dF = dice_np(seg_probs(net, XF) > .5, MF > 0); pn = seg_probs(net, XN)
    r = dict(dice_figshare_new_patients=float(dF.mean()), dice_median=float(np.median(dF)),
             dice_by_class={CLASSES[k]: float(dF[fy[s_te] == k].mean()) for k in range(3) if (fy[s_te] == k).any()},
             healthy_false_contour_area05=float(((pn > .5).mean((1, 2)) > .005).mean()),
             healthy_false_contour_demo=float(np.mean([demo_contour(p) for p in pn])))
    if XO is not None: r['dice_org_images'] = float(dice_np(seg_probs(net, XO) > .5, MO > 0).mean())
    return r

def run_seg(tag, what, use_org=True, use_empty=True, seed=42, epochs=20):
    if tag in R: print(f'[{tag}] уже готов: Dice {R[tag]["test"]["dice_figshare_new_patients"]:.4f}'); return
    set_seed(seed)
    items = [('f', i) for i in s_tr] + ([('o', q) for q in q_tr] if use_org else []) + ([('n', i) for i in TR if LAB[i] == 3] if use_empty else [])
    net = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=1, classes=1).to(dev)
    sdl = DataLoader(SDS(items, True), 16, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    opt = torch.optim.AdamW(net.parameters(), 3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, total_steps=epochs * len(sdl), pct_start=.1)
    scaler = torch.cuda.amp.GradScaler(); dl_ = smp.losses.DiceLoss('binary'); bce = nn.BCEWithLogitsLoss()
    CK, BEST = W/'ckpt'/f'exp_{tag}_last.pt', W/'ckpt'/f'exp_{tag}_best.pt'
    start, best = 0, -1.0
    if CK.exists():
        c = torch.load(CK, map_location=dev, weights_only=False)
        net.load_state_dict(c['m']); opt.load_state_dict(c['o']); sch.load_state_dict(c['s']); scaler.load_state_dict(c['sc'])
        start, best = c['ep'] + 1, c['best']; print(f'[{tag}] продолжаю с эпохи', start + 1)
    print(f'[{tag}] обучающих примеров: {len(items)}')
    for ep in range(start, epochs):
        net.train(); t0 = time.time(); tl = 0
        for x, y in sdl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda'):
                o = net(x).float(); loss = dl_(o, y) + bce(o, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()
        d = dice_np(seg_probs(net, XV) > .5, MV > 0).mean()
        if d > best: best = d; torch.save(net.state_dict(), BEST)
        torch.save(dict(m=net.state_dict(), o=opt.state_dict(), s=sch.state_dict(), sc=scaler.state_dict(), ep=ep, best=best), CK)
        print(f'[{tag}] эпоха {ep+1}/{epochs}: loss {tl/len(sdl):.3f}  val Dice {d:.4f}  лучший {best:.4f}  {time.time()-t0:.0f}с')
    net.load_state_dict(torch.load(BEST, map_location=dev))
    R[tag] = dict(what=what, use_org=use_org, use_empty=use_empty, seed=seed, epochs=epochs, n_train=len(items),
                  val_dice=float(best), test=seg_eval(net))
    save_R(); CK.unlink(missing_ok=True)
    print(f'[{tag}] ТЕСТ:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in R[tag]['test'].items()})
    del net, opt; torch.cuda.empty_cache()

if 'seg_base' not in R:
    nb_ = smp.Unet('resnet34', encoder_weights=None, in_channels=1, classes=1).to(dev)
    nb_.load_state_dict(torch.load(W/'ckpt'/'seg_v3_best.pt', map_location=dev))
    R['seg_base'] = dict(what='Основная сегментация: Figshare + маски на снимках организаторов + пустые маски на здоровых',
                         use_org=True, use_empty=True, seed=42, epochs=20, test=seg_eval(nb_))
    save_R(); del nb_; torch.cuda.empty_cache()
if (W/'results_seg_v3.json').exists():
    ref = json.load(open(W/'results_seg_v3.json'))['dice_mean']
    print(f'Сверка с основным блокнотом: Dice {R["seg_base"]["test"]["dice_figshare_new_patients"]:.4f} (было {ref:.4f})')
print('Основная сегментация:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in R['seg_base']['test'].items()})
safe(run_seg, 'seg_no_org', 'Без перенесённых масок на снимках организаторов (только Figshare + здоровые)', use_org=False)
safe(run_seg, 'seg_no_empty', 'Без здоровых снимков с пустой маской (не учим «не рисовать»)', use_empty=False)

In [ ]:
# === E4. ИТОГОВАЯ ТАБЛИЦА (сохраняется в Drive: experiments_v1.md) ===
def tbl(d):
    try: return d.round(4).to_markdown(index=False)
    except ImportError: return d.round(4).to_string(index=False)
cls_keys = [k for k in ['base', 'seed1', 'seed2', 'class_w', 'epochs30', 'ens_seeds', 'ens_all', 'no_aug', 'resnet50', 'effb3', 'no_pretrain', 'size128'] if k in R]
b = R['base']['test']; rows = []
for k in cls_keys:
    t = R[k]['test']
    rows.append({'эксперимент': R[k]['what'], 'val macro-F1': R[k].get('val_macro_f1', np.nan), 'accuracy': t['accuracy'], 'Δ к основной, п.п.': 100 * (t['accuracy'] - b['accuracy']),
                 'macro-F1': t['macro_f1'], 'чувств.': t['tumor_sens'], 'спец.': t['tumor_spec'], 'recall глиомы': t['glioma_recall'], 'AUC': t['tumor_auc']})
T1 = pd.DataFrame(rows)
seeds = [R[k]['test'] for k in ['base', 'seed1', 'seed2'] if k in R]
lines = ['# Эксперименты (честный групповой тест, то же разбиение, что в основной работе)', '',
         f'Тест: {b["n"]} снимков. Каждый эксперимент меняет одну вещь относительно основного рецепта.', '',
         '## Классификатор', '', tbl(T1), '']
if 'selected' in R:
    lines += [f'**Выбор финальной модели — по валидации (macro-F1), не по тесту:** {R["selected"]["tag"]}', '']
if len(seeds) >= 2:
    S = {m_: (np.mean([s[m_] for s in seeds]), np.std([s[m_] for s in seeds], ddof=1)) for m_ in ['accuracy', 'macro_f1', 'tumor_sens', 'tumor_spec', 'glioma_recall']}
    R['seed_summary'] = {m_: dict(mean=float(v[0]), std=float(v[1]), n=len(seeds)) for m_, v in S.items()}; save_R()
    lines += [f'**Повторы обучения ({len(seeds)} seed):** ' + ', '.join(f'{m_} {v[0]:.4f} ± {v[1]:.4f}' for m_, v in S.items()), '']
seg_keys = [k for k in ['seg_base', 'seg_no_org', 'seg_no_empty'] if k in R]
if seg_keys:
    T2 = pd.DataFrame([{'эксперимент': R[k]['what'], 'Dice (новые пациенты)': R[k]['test']['dice_figshare_new_patients'],
                        'Dice (снимки организаторов)': R[k]['test'].get('dice_org_images', np.nan),
                        'Dice глиома': R[k]['test']['dice_by_class'].get('Глиома', np.nan),
                        'ложный контур на здоровых (правило демо)': R[k]['test']['healthy_false_contour_demo'],
                        'ложный контур (площадь > 0.5%)': R[k]['test']['healthy_false_contour_area05']} for k in seg_keys])
    lines += ['## Сегментация', '', tbl(T2), '']
md_txt = '\n'.join(lines); open(W/'experiments_v1.md', 'w').write(md_txt); print(md_txt)
print('\nСохранено в Drive: almazov/experiments_v1.json и almazov/experiments_v1.md — пришли их мне.')